**Descripción:** Este notebook contiene el flujo de trabajo completo para transformar el código original de Deep Learning (series temporales) en una arquitectura MLOps profesional.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
pip install wandb fastai

In [ ]:
!pip install dvc dvc-gdrive

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.1/470.1 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.2/451.2 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.2/214.2 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9

In [ ]:
!git init
!dvc init

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/
/bin/bash: line 1: dvc: command not found


In [ ]:
!dvc init -f

Initialized DVC repository.

You can now commit the changes to git.

+---------------------------------------------------------------------+
|                                                                     |
|        DVC has enabled anonymous aggregate usage analytics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|             <https://dvc.org/doc/user-guide/analytics>              |
|                                                                     |
+---------------------------------------------------------------------+

What's next?
------------
- Check out the documentation: <https://dvc.org/doc>
- Get help and share ideas: <https://dvc.org/chat>
- Star us on GitHub: <https://github.com/treeverse/dvc>


In [ ]:

!dvc remote add -d myremote gdrive://D0RfaBOev5rdINOSzZXoSm0lxAPx-0jV?usp=drive_link
!dvc remote modify myremote gdrive_use_service_account true # Opcional según permisos

/bin/bash: line 1: dvc: command not found
/bin/bash: line 1: dvc: command not found


In [ ]:
import os

# Definir la ruta de datos
data_dir = '/content/data'

if not os.path.exists(data_dir):
    os.makedirs(data_dir)
    print(f"Carpeta {data_dir} creada con éxito.")

Carpeta /content/data creada con éxito.


In [ ]:
import kagglehub
import shutil
import os

# 1. Descargar el dataset
path_descarga = kagglehub.dataset_download("shayanfazeli/heartbeat")

# 2. Asegurarnos de que la carpeta de destino existe
data_dir = '/content/data'
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

# 3. Mover los archivos específicos que necesitamos
archivos_a_mover = ['mitbih_train.csv', 'mitbih_test.csv']

for archivo in archivos_a_mover:
    origen = os.path.join(path_descarga, archivo)
    destino = os.path.join(data_dir, archivo)
    if os.path.exists(origen):
        # Cambiado de shutil.move a shutil.copy porque el origen es un sistema de archivos de solo lectura
        shutil.copy(origen, destino)
        print(f"Copiado: {archivo} a {data_dir}")
    else:
        print(f"Error: No se encontró {archivo} en la descarga.")

# 4. Verificar qué hay en la carpeta ahora
print("Contenido de /content/data:", os.listdir(data_dir))

100%|██████████| 98.8M/98.8M [00:00<00:00, 152MB/s]

Extracting files...


Copiado: mitbih_train.csv a /content/data
Copiado: mitbih_test.csv a /content/data
Contenido de /content/data: ['mitbih_test.csv', 'mitbih_train.csv']


In [ ]:
# los datos están en la carpeta 'data/'
!dvc add data/mitbih_train.csv data/mitbih_test.csv

⠋ Checking graph
Adding...:   0% 0/2 [00:00<?, ?file/s{'info': ' data/mitbih_train.csv |'}]
!
          |0.00 [00:00,     ?file/s]
                                    
!
  0% |          |0/? [00:00<?,    ?files/s]
                                           
Adding data/mitbih_train.csv to cache:   0% 0/1 [00:00<?, ?file/s]
Adding data/mitbih_train.csv to cache:   0% 0/1 [00:00<?, ?file/s{'info': ''}]
100% 1/1 [00:00<00:00,  1.36file/s{'info': ''}]                               
                                               
Checking out /content/data/mitbih_train.csv:   0% 0/1 [00:00<?, ?files/s]
  0% 0/1 [00:00<?, ?files/s{'info': ''}]                                 
100% 1/1 [00:03<00:00,  3.21s/files{'info': ''}]
 50% 1/2 [00:05<00:05,  5.09s/file{'info': ' data/mitbih_test.csv |'}] 
!
          |0.00 [00:00,     ?file/s]
                                    
!
  0% |          |0/? [00:00<?,    ?files/s]
                                           
Adding data/mitbih_test.csv to cac

In [ ]:
import os

# Definir la base del proyecto
project_path = '/content/drive/MyDrive/Proyecto_MLOps_ECG'

# Lista de carpetas necesarias para cumplir con el estándar MLOps
folders = [
    'src',
    'api',
    'notebooks',
    'data',  #no creo que lo subo a Github
    'models',
    'tests'
]

for folder in folders:
    full_path = os.path.join(project_path, folder)
    if not os.path.exists(full_path):
        os.makedirs(full_path)
        print(f"Creada carpeta: {full_path}")
    else:
        print(f"La carpeta ya existe: {full_path}")

La carpeta ya existe: /content/drive/MyDrive/Proyecto_MLOps_ECG/src
La carpeta ya existe: /content/drive/MyDrive/Proyecto_MLOps_ECG/api
La carpeta ya existe: /content/drive/MyDrive/Proyecto_MLOps_ECG/notebooks
La carpeta ya existe: /content/drive/MyDrive/Proyecto_MLOps_ECG/data
La carpeta ya existe: /content/drive/MyDrive/Proyecto_MLOps_ECG/models
La carpeta ya existe: /content/drive/MyDrive/Proyecto_MLOps_ECG/tests


In [ ]:
%%writefile /content/drive/MyDrive/Proyecto_MLOps_ECG/src/train.py
import wandb
import torch
import sys
import numpy as np
from fastai.vision.all import *
from fastai.callback.wandb import WandbCallback
from pathlib import Path

# Configuración de rutas
root = Path("/content/drive/MyDrive/Proyecto_MLOps_ECG").resolve()
sys.path.append(str(root))

from src.model import InceptionTime
from src.data_loader import ECGDataLoader
from src.utils import preprocess_signal

def train_inception(config=None):
    # Inicializar wandb
    if config is None:
        config = {}

    run = wandb.init(config=config)
    config = wandb.config

    print("Configuración:", dict(config))

    # Carga de datos
    print("Cargando datos...")
    loader = ECGDataLoader(target_samples=int(config.target_samples))
    loader.download_data()
    train_df, test_df = loader.load_and_balance()

    print("Preprocesando...")
    X_train, y_train = preprocess_signal(train_df)
    X_test, y_test = preprocess_signal(test_df)

    print(f"Shape de datos: Train={X_train.shape}, Test={X_test.shape}")

    # Convertir a tensores y crear datasets para fastai
    # Fastai puede trabajar directamente con tensores de PyTorch
    train_x = torch.FloatTensor(X_train)
    train_y = torch.LongTensor(y_train)
    test_x = torch.FloatTensor(X_test)
    test_y = torch.LongTensor(y_test)

    # Crear DataLoaders usando el método de fastai

    class CustomDataset:
        def __init__(self, x, y):
            self.x = x
            self.y = y

        def __len__(self):
            return len(self.x)

        def __getitem__(self, idx):
            return self.x[idx], self.y[idx]

    train_ds = CustomDataset(train_x, train_y)
    test_ds = CustomDataset(test_x, test_y)

    # DataLoaders
    dls = DataLoaders.from_dsets(
        train_ds, test_ds,
        bs=int(config.batch_size),
        num_workers=0
    )

    if torch.cuda.is_available():
        dls.cuda()
        print("Usando GPU")
    else:
        print("Usando CPU")

    # Modelo
    model = InceptionTime(n_classes=5, nf=int(config.nf))
    if torch.cuda.is_available():
        model = model.cuda()

    # Learner con WandbCallback
    learn = Learner(
        dls,
        model,
        loss_func=LabelSmoothingCrossEntropy(),
        metrics=[accuracy],
        cbs=[WandbCallback(log_preds=False, log_model=True)]
    )

    # Entrenar
    print(f"Iniciando entrenamiento por {config.epochs} épocas...")
    learn.fit_one_cycle(int(config.epochs), float(config.lr))

    # Guardar modelo
    model_path = Path("/content/drive/MyDrive/Proyecto_MLOps_ECG/models/best_model.pth")
    model_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), model_path)
    print(f"Modelo guardado en {model_path}")

    # Guardar modelo en wandb
    wandb.save(str(model_path))

    # Mostrar resultados finales
    if len(learn.recorder.values) > 0:
        final_metrics = learn.recorder.values[-1]
        print(f"\n📊 Resultados finales:")
        print(f"  - Valid Loss: {final_metrics[0]:.4f}")
        print(f"  - Accuracy: {final_metrics[1]:.4f}")

    run.finish()
    print("✅ Entrenamiento completado!")

if __name__ == "__main__":
    # Configuración por defecto
    default_config = {
        "epochs": 10,
        "batch_size": 32,
        "lr": 0.001,
        "nf": 64,
        "target_samples": 500
    }
    train_inception(default_config)

Overwriting /content/drive/MyDrive/Proyecto_MLOps_ECG/src/train.py


In [ ]:
%%writefile /content/drive/MyDrive/Proyecto_MLOps_ECG/src/utils.py
import numpy as np
import pandas as pd
from sklearn.utils import resample

def preprocess_signal(df):
    """
    Convierte el DataFrame en tensores de PyTorch
    """
    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values
    X = X.reshape(X.shape[0], 1, X.shape[1])
    return X.astype(np.float32), y.astype(np.int64)

def balance_data(df, n_samples=20000):
    """
    Aplica el balanceo de clases que usaste en tu práctica
    """
    df_0 = df[df[187] == 0]
    df_1 = df[df[187] == 1]
    df_2 = df[df[187] == 2]
    df_3 = df[df[187] == 3]
    df_4 = df[df[187] == 4]

    # Remuestreo para equilibrar las clases a n_samples
    df_0_res = resample(df_0, replace=True, n_samples=n_samples, random_state=42)
    df_1_res = resample(df_1, replace=True, n_samples=n_samples, random_state=42)
    df_2_res = resample(df_2, replace=True, n_samples=n_samples, random_state=42)
    df_3_res = resample(df_3, replace=True, n_samples=n_samples, random_state=42)
    df_4_res = resample(df_4, replace=True, n_samples=n_samples, random_state=42)

    return pd.concat([df_0_res, df_1_res, df_2_res, df_3_res, df_4_res])

Overwriting /content/drive/MyDrive/Proyecto_MLOps_ECG/src/utils.py


In [ ]:
%%writefile /content/drive/MyDrive/Proyecto_MLOps_ECG/src/model.py
import torch
import torch.nn as nn

class InceptionModule(nn.Module):
    def __init__(self, ni, nf, ks=[9, 19, 39], bottleneck=True):
        super().__init__()
        self.ks = ks
        self.bottleneck = nn.Conv1d(ni, nf, 1, bias=False) if bottleneck else nn.Identity()
        self.convs = nn.ModuleList([nn.Conv1d(nf if bottleneck else ni, nf, k, padding=k//2, bias=False) for k in ks])
        self.maxpool = nn.Sequential(nn.MaxPool1d(3, stride=1, padding=1), nn.Conv1d(ni, nf, 1, bias=False))
        self.bn = nn.BatchNorm1d(nf * 4)
        self.act = nn.ReLU()

    def forward(self, x):
        input_tensor = x
        x = self.bottleneck(x)
        out = torch.cat([c(x) for c in self.convs] + [self.maxpool(input_tensor)], dim=1)
        return self.act(self.bn(out))

class InceptionTime(nn.Module):
    def __init__(self, n_classes=5, nf=32):
        super().__init__()
        self.block1 = InceptionModule(1, nf)
        self.block2 = InceptionModule(nf * 4, nf)
        self.adaptive_pool = nn.AdaptiveAvgPool1d(1)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(nf * 4, n_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.adaptive_pool(x)
        x = self.flatten(x)
        return self.fc(x)

Overwriting /content/drive/MyDrive/Proyecto_MLOps_ECG/src/model.py


In [ ]:
%%writefile /content/drive/MyDrive/Proyecto_MLOps_ECG/src/data_loader.py
import pandas as pd
import kagglehub
import os
import shutil
from .utils import balance_data

class ECGDataLoader:
    def __init__(self, target_samples=20000, data_dir='/content/data'):
        self.target_samples = target_samples
        self.data_dir = data_dir
        self.train_path = os.path.join(data_dir, 'mitbih_train.csv')
        self.test_path = os.path.join(data_dir, 'mitbih_test.csv')

    def download_data(self):
        """Descarga desde Kaggle y organiza en la carpeta local"""
        if not os.path.exists(self.train_path):
            print("Descargando dataset de Kaggle...")
            path = kagglehub.dataset_download("shayanfazeli/heartbeat")

            if not os.path.exists(self.data_dir):
                os.makedirs(self.data_dir)

            shutil.copy(os.path.join(path, 'mitbih_train.csv'), self.train_path)
            shutil.copy(os.path.join(path, 'mitbih_test.csv'), self.test_path)
            print("Datos listos en:", self.data_dir)
        else:
            print("Los datos ya existen localmente.")

    def load_and_balance(self):
        """Carga los CSV y aplica el balanceo de clases"""
        train_df = pd.read_csv(self.train_path, header=None)
        test_df = pd.read_csv(self.test_path, header=None)

        print(f"Balanceando clases a {self.target_samples} muestras...")
        train_balanced = balance_data(train_df, n_samples=self.target_samples)

        return train_balanced, test_df

Overwriting /content/drive/MyDrive/Proyecto_MLOps_ECG/src/data_loader.py


In [ ]:
import wandb

sweep_config = {
    'method': 'grid', # Para probar combinaciones específicas
    'metric': {'name': 'accuracy', 'goal': 'maximize'},
    'parameters': {
        'nf': {'values': [16, 32]}, # Solo dos para ir rápido
        'lr': {'values': [1e-2, 1e-3]}, # dos velocidades de aprendizaje
        'epochs': {'value': 3}, # Muy pocas épocas para ver la curva rápido
        'batch_size': {'value': 128}, # Batch más grande para acelerar el entrenamiento
        'target_samples': {'value': 2000} # Suficiente para ver tendencias
    }
}

# Crear el nuevo ID del Sweep
sweep_id = wandb.sweep(sweep_config, project="ECG-MLOps-Project")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Create sweep with ID: 77nkt8p2
Sweep URL: https://wandb.ai/ali-adib-csic/ECG-MLOps-Project/sweeps/77nkt8p2


In [ ]:
from pathlib import Path
import sys

# 1. Definimos la ruta base (Root) (en Colab a veces saltan mucha probelmas con la ruta)

base_path = Path("/content/drive/MyDrive/Proyecto_MLOps_ECG").resolve()

# 2. Verificar que la carpeta existe
if base_path.exists():

    if str(base_path) not in sys.path:
        sys.path.append(str(base_path))
    print(f"✅ Proyecto detectado en: {base_path}")
else:
    print("❌ Error: La ruta no existe. ¿Has montado el Drive?")

# 3. Ahora las importaciones son seguras
from src.train import train_inception

✅ Proyecto detectado en: /content/drive/MyDrive/Proyecto_MLOps_ECG


In [ ]:
import wandb
import sys
import torch


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Entrenando en: {device}")

# Lanzar el  Sweep con la referencia correspondiente

wandb.agent("77nkt8p2", function=train_inception, project="ECG-MLOps-Project", count=3)

🚀 Entrenando en: cpu


wandb: Agent Starting Run: 17z8o50v with config:
wandb: 	batch_size: 128
wandb: 	epochs: 3
wandb: 	lr: 0.01
wandb: 	nf: 16
wandb: 	target_samples: 2000
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: ali-adib (ali-adib-csic) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Configuración: {'batch_size': 128, 'epochs': 3, 'lr': 0.01, 'nf': 16, 'target_samples': 2000}
Cargando datos...
Los datos ya existen localmente.
Balanceando clases a 2000 muestras...
Preprocesando...
Shape de datos: Train=(10000, 1, 187), Test=(21892, 1, 187)
Usando CPU
Iniciando entrenamiento por 3 épocas...


epoch,train_loss,valid_loss,accuracy,time
0,1.250736,1.256682,0.450256,00:56
1,0.954321,1.232863,0.533254,00:54
2,0.834036,0.975458,0.639503,00:58


Better model found at epoch 0 with valid_loss value: 1.2566819190979004.
Better model found at epoch 1 with valid_loss value: 1.2328630685806274.
Better model found at epoch 2 with valid_loss value: 0.975458025932312.


...

Better model found at epoch 0 with valid_loss value: 1.598362684249878.
Better model found at epoch 1 with valid_loss value: 1.08551824092865.


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Modelo guardado en /content/drive/MyDrive/Proyecto_MLOps_ECG/models/best_model.pth

📊 Resultados finales:
  - Valid Loss: 0.9854
  - Accuracy: 1.1423


accuracy,▁█▆
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇██
eps_0,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
lr_0,▁▁▂▂▂▃▃▄▄▆▇████▇▇▇▆▆▆▅▅▅▅▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁
mom_0,██▇▇▇▆▆▅▅▄▂▁▁▁▁▁▁▂▂▂▃▃▃▃▃▄▄▄▅▆▆▆▇▇▇▇████
raw_loss,███▇▇▆▇▆▆▆▆▅▅▅▅▅▅▄▄▄▄▃▄▄▃▂▂▃▃▂▁▁▂▁▂▁▁▂▂▁
sqr_mom_0,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,████▇▇▇▇▇▆▆▅▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
train_samples_per_sec,█▂▁▁▇██▂▂▂█▇▇█▇█▇█▇█▃▇█▇▂▇▆██▁▂▂▂▁▁▇██▇▂
valid_loss,█▁▂
+1,...


✅ Entrenamiento completado!


In [ ]:
%%writefile /content/drive/MyDrive/Proyecto_MLOps_ECG/api/main.py
import os
import torch
import numpy as np
import wandb
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List

# Importar la arquitectura y preprocesamiento de la carpeta src
from src.model import InceptionTime
from src.utils import preprocess_signal

app = FastAPI(
    title="ECG MLOps Service",
    description="Servicio de clasificación de arritmias cardíacas usando InceptionTime"
)

# --- CONFIGURACIÓN DE W&B ---
ENTITY = "ali-adib-csic"
PROJECT = "ECG-MLOps-Project"
MODEL_ARTIFACT = f"{ENTITY}/{PROJECT}/model:v2"

# Variable global para el modelo
model_engine = None

class ECGRequest(BaseModel):
    # El usuario envía una lista de 187 valores (longitud estándar del MIT-BIH)
    signal: List[float]

@app.on_event("startup")
def load_candidate_model():
    """Descarga el artefacto v2 de W&B y reconstruye el modelo."""
    global model_engine
    try:
        print(f"Descargando artefacto: {MODEL_ARTIFACT}...")
        run = wandb.init(project=PROJECT, job_type="inference")
        artifact = run.use_artifact(MODEL_ARTIFACT, type='model')
        artifact_dir = artifact.download()

        # 1. Instanciar la arquitectura

        model_engine = InceptionTime(n_classes=5, nf=64)

        # 2. Cargar los pesos
        # Buscar el archivo .pth en el directorio descargado
        path_weights = next(iter([f for f in os.listdir(artifact_dir) if f.endswith('.pth')]), None)

        if path_weights:
            state_dict = torch.load(os.path.join(artifact_dir, path_weights), map_location='cpu')
            model_engine.load_state_dict(state_dict)
            model_engine.eval()
            print("✅ Modelo cargado y listo para inferencia.")
        else:
            raise FileNotFoundError("No se encontró archivo .pth en el artefacto.")

        run.finish()
    except Exception as e:
        print(f"❌ Error crítico al cargar modelo: {e}")

@app.get("/")
def health_check():
    return {"status": "online", "model": MODEL_ARTIFACT}

@app.post("/predict")
async def predict(request: ECGRequest):
    if model_engine is None:
        raise HTTPException(status_code=503, detail="Modelo no disponible")

    try:
        # 1. Convertir entrada a DataFrame temporal para reusar el preprocess_signal

        import pandas as pd
        temp_df = pd.DataFrame([request.signal + [0]])

        # 2. Preprocesar (X tendrá forma 1, 1, 187)
        X, _ = preprocess_signal(temp_df)
        input_tensor = torch.from_numpy(X).float()

        # 3. Inferencia
        with torch.no_grad():
            output = model_engine(input_tensor)
            # InceptionTime devuelve (fc_output, )
            logits = output if isinstance(output, torch.Tensor) else output[0]

            prob = torch.softmax(logits, dim=1)
            pred_class = torch.argmax(prob, dim=1).item()
            confidence = prob[0][pred_class].item()

        # Mapeo de clases del MIT-BIH
        classes = {0: "Normal", 1: "S", 2: "V", 3: "F", 4: "Q"}

        return {
            "prediction": classes.get(pred_class, "Unknown"),
            "class_index": pred_class,
            "confidence": round(confidence, 4)
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Error en inferencia: {str(e)}")

Writing /content/drive/MyDrive/Proyecto_MLOps_ECG/api/main.py


In [ ]:
pip install torch

In [ ]:
%%writefile /content/drive/MyDrive/Proyecto_MLOps_ECG/src/train_rocket.py
import wandb
import numpy as np
from sklearn.linear_model import RidgeClassifierCV
from sktime.transformations.panel.rocket import Rocket
from pathlib import Path
import sys

# Configuración de rutas
root = Path("/content/drive/MyDrive/Proyecto_MLOps_ECG").resolve()
sys.path.append(str(root))

from src.data_loader import ECGDataLoader
from src.utils import preprocess_signal

def train_rocket():
    # Inicializar wandb para el Baseline
    run = wandb.init(project="ECG-MLOps-Project", job_type="baseline")

    # Configuración fija para el Baseline
    config = {
        "num_kernels": 1000, # ROCKET usa kernels aleatorios
        "target_samples": 500 #  lo mismo que el sweep rápido para comparar
    }
    wandb.config.update(config)

    # 1. Carga de datos
    loader = ECGDataLoader(target_samples=config["target_samples"])
    loader.download_data()
    train_df, test_df = loader.load_and_balance()

    X_train, y_train = preprocess_signal(train_df)
    X_test, y_test = preprocess_signal(test_df)

    # ROCKET espera (n_instances, n_columns, n_timepoints)


    print(f"🚀 Iniciando ROCKET con {config['num_kernels']} kernels...")

    # 2. Transformación ROCKET
    rocket = Rocket(num_kernels=config["num_kernels"])
    rocket.fit(X_train)
    X_train_transform = rocket.transform(X_train)

    # 3. Clasificador
    classifier = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
    classifier.fit(X_train_transform, y_train)

    # 4. Evaluación
    X_test_transform = rocket.transform(X_test)
    accuracy = classifier.score(X_test_transform, y_test)

    print(f"✅ ROCKET Accuracy: {accuracy:.4f}")

    # 5. Log a W&B
    wandb.log({"accuracy": accuracy, "model_type": "ROCKET_Baseline"})
    run.finish()

if __name__ == "__main__":
    train_rocket()

Writing /content/drive/MyDrive/Proyecto_MLOps_ECG/src/train_rocket.py


In [ ]:
!pip install sktime
import sys
from pathlib import Path


project_root = Path("/content/drive/MyDrive/Proyecto_MLOps_ECG").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.train_rocket import train_rocket

# Ejecutar el baseline
train_rocket()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 20.0 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ali-adib (ali-adib-csic) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Los datos ya existen localmente.
Balanceando clases a 500 muestras...
🚀 Iniciando ROCKET con 1000 kernels...
✅ ROCKET Accuracy: 0.8735


accuracy,▁
accuracy,0.87347
model_type,ROCKET_Baseline


In [ ]:
%%writefile /content/drive/MyDrive/Proyecto_MLOps_ECG/requirements.txt
fastapi==0.110.0
uvicorn==0.27.1
--extra-index-url https://download.pytorch.org/whl/cpu
torch==2.2.1
torchvision==0.17.1
scikit-learn==1.4.1.post1
pandas==2.2.0
numpy==1.26.4
wandb==0.16.3
pydantic==2.6.1
python-multipart==0.0.9
httpx==0.27.0
pytest==8.0.0

Writing /content/drive/MyDrive/Proyecto_MLOps_ECG/requirements.txt


In [ ]:
%%writefile /content/drive/MyDrive/Proyecto_MLOps_ECG/.dockerignore

# Git
.git
.gitignore

# Python temporales
__pycache__/
*.py[cod]
*$py.class
.pytest_cache/
.venv
venv/
env/

# Datos y Modelos locales (ya se bajan de W&B o Kaggle en el contenedor)
data/
models/*.pth
artifacts/

# Google Drive / Colab (específico de la estructura actual)
.ipynb_checkpoints/
*.ipynb

# Variables de entorno y secretos
.env
.wandb
wandb/

Overwriting /content/drive/MyDrive/Proyecto_MLOps_ECG/.dockerignore


In [ ]:
%%writefile /content/drive/MyDrive/Proyecto_MLOps_ECG/Dockerfile

# Imagen base ligera
FROM python:3.9-slim

# Evita que Python genere archivos .pyc y permite ver logs en tiempo real
ENV PYTHONDONTWRITEBYTECODE 1
ENV PYTHONUNBUFFERED 1

WORKDIR /app

# Instalar dependencias del sistema necesarias
RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential && \
    rm -rf /var/lib/apt/lists/*

# Copiar requirements
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copiamos el código respetando tu estructura de Drive
COPY ./src ./src
COPY ./api ./api

# Exponemos el puerto 8000
EXPOSE 8000

# Comando para arrancar la API
CMD ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"]


Writing /content/drive/MyDrive/Proyecto_MLOps_ECG/Dockerfile


In [ ]:
%%writefile /content/drive/MyDrive/Proyecto_MLOps_ECG/src/__init__.py


Overwriting /content/drive/MyDrive/Proyecto_MLOps_ECG/src/__init__.py
